In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("final_cleaned_user_data.csv")

In [3]:
df.columns

Index(['firstName', 'lastName', 'handle', 'rating', 'avg_difficulty',
       'contest_count', 'title', 'accuracy', 'total_problems_solved'],
      dtype='object')

In [4]:
CF_COLOURS = {
    'newbie':             '#808080',
    'pupil':              '#008000',
    'specialist':         '#03A89E',
    'expert':             '#0000FF',
    'candidate master':   '#AA00AA',
    'master':             '#FF8C00',
    'international master': '#FF8C00',
    'grandmaster':        '#FF0000',
    'international grandmaster':'#FF0000',
    'legendary grandmaster':'#FF0000',
}

In [5]:
import plotly.express as px

In [6]:
# Colored BoxPlot (on the basis of rank) y -> avg_difficulty x -> rank/title using plotly express

fig = px.box(
    df,
    x="title",
    y="avg_difficulty",
    title="Avg Problem Difficulty by title",
    color="title",
    color_discrete_map=CF_COLOURS
)

fig.update_layout(
    xaxis_title="Rank",
    yaxis_title="Average Difficulty of Solved Problems",
    hovermode="x unified",
)

fig.show()

In [7]:
#Getting all the 5 point summary for each title, give the code

summary = df.groupby('title')['avg_difficulty'].describe()
print(summary)

                           count         mean         std          min  \
title                                                                    
candidate master            50.0  1398.644171  160.727830  1036.363636   
expert                      50.0  1261.146466  151.148924   953.551913   
grandmaster                 31.0  1679.405510  157.374220  1427.659574   
international grandmaster    5.0  1729.338657   46.475292  1689.844852   
international master        32.0  1589.091712  112.544163  1374.054759   
legendary grandmaster        1.0  1792.140266         NaN  1792.140266   
master                      50.0  1455.198880  131.105665  1225.125628   
newbie                      50.0   926.141239  133.960746   800.000000   
pupil                       50.0  1029.713819  138.889000   800.000000   
specialist                  50.0  1138.189087  140.879197   905.128205   

                                   25%          50%          75%          max  
title                          

In [8]:
fig = px.scatter(
    df,
    x="rating",
    y="avg_difficulty",
    color="title",
    color_discrete_map=CF_COLOURS,
    title="Do Top Coders Solve Harder Problems?",
    hover_data=["handle", "contest_count"]
)
fig.show()

In [9]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

titles = sorted(df['title'].dropna().unique().tolist())

title_select = widgets.SelectMultiple(
    options=titles,
    value=tuple(titles[:2]) if len(titles) >= 2 else tuple(titles),
    description='Titles',
    rows=min(10, len(titles))
)

out = widgets.Output()

def draw_chart(selected_titles):
    selected_titles = list(selected_titles)

    with out:
        clear_output(wait=True)

        if not selected_titles:
            print("Select at least one title.")
            return

        sub = df[df['title'].isin(selected_titles)].copy()

        fig = px.scatter(
            sub,
            x="contest_count",
            y="avg_difficulty",
            size="total_problems_solved",
            color="title",
            color_discrete_map=CF_COLOURS,
            hover_data=[
                "handle",
                "contest_count",
                "avg_difficulty",
                "total_problems_solved"
            ],
            title="Compare Selected Titles on One Scatter Plot",
            labels={
                "contest_count": "Contests Participated",
                "avg_difficulty": "Avg Problem Difficulty",
                "total_problems_solved": "Total Problems Solved"
            },
            size_max=30,
        )

        fig.update_traces(
            marker=dict(opacity=0.75, line=dict(width=0.7, color="white"))
        )

        fig.update_xaxes(range=[0, df["contest_count"].max() * 1.05])
        fig.update_yaxes(range=[0, df["avg_difficulty"].max() * 1.05])

        fig.show()

def on_change(change):
    if change["name"] == "value":
        draw_chart(change["new"])

title_select.observe(on_change)

display(title_select, out)
draw_chart(title_select.value)

SelectMultiple(description='Titles', index=(0, 1), options=('candidate master', 'expert', 'grandmaster', 'inte…

Output()

In [10]:
correlation_data = df[['rating', 'contest_count', 'avg_difficulty', 'total_problems_solved']].corr()

fig = px.imshow(
    correlation_data,
    labels=dict(x="Metric", y="Metric", color="Correlation"),
    title="Correlation: Rating vs Performance Factors",
    color_continuous_scale="viridis",
    zmin=-1, zmax=1,
    text_auto='.2f'
)
fig.show()

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.express as px

titles = sorted(df["title"].dropna().unique().tolist())

title_select = widgets.SelectMultiple(
    options=titles,
    value=tuple(titles[:2]) if len(titles) >= 2 else tuple(titles),
    description="Titles",
    rows=min(10, len(titles)),
)

out = widgets.Output()

def draw_3d(selected_titles):
    selected_titles = list(selected_titles)

    with out:
        clear_output(wait=True)

        if not selected_titles:
            print("Select at least one title.")
            return

        sub = df[df["title"].isin(selected_titles)].copy()

        fig = px.scatter_3d(
            sub,
            x="contest_count",
            y="avg_difficulty",
            z="total_problems_solved",
            color="title",
            size="total_problems_solved",
            color_discrete_map=CF_COLOURS,
            hover_data=[
                "handle",
                "contest_count",
                "avg_difficulty",
                "total_problems_solved",
                "rating",
            ],
            title="3D Comparison: Contests × Difficulty × Problems Solved",
            labels={
                "contest_count": "Contests Participated",
                "avg_difficulty": "Avg Problem Difficulty",
                "total_problems_solved": "Total Problems Solved",
            },
            size_max=30,
        )

        fig.update_traces(
            marker=dict(opacity=0.75, line=dict(width=0.6, color="white"))
        )

        fig.update_layout(
            height=900,
            width=1200,
            legend_title_text="Title",
            margin=dict(l=0, r=0, t=60, b=0),
            scene=dict(
                aspectmode="cube",
                xaxis_title="Contests Participated",
                yaxis_title="Avg Problem Difficulty",
                zaxis_title="Total Problems Solved",
            ),
        )

        fig.show(config={"scrollZoom": True})

def on_change(change):
    if change["name"] == "value":
        draw_3d(change["new"])

title_select.observe(on_change)

display(title_select, out)
draw_3d(title_select.value)

SelectMultiple(description='Titles', index=(0, 1), options=('candidate master', 'expert', 'grandmaster', 'inte…

Output()

In [12]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

In [13]:
summary = df.groupby('title').agg({
    'contest_count': ['mean', 'median'],
    'avg_difficulty': ['mean', 'median'],
    'total_problems_solved': ['mean', 'median'],
    'rating': 'count'  # user count
}).round(0)

print(summary)

# Create side-by-side bars
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Contests", "Avg Difficulty", "Problems Solved")
)

title_order = ['newbie', 'pupil', 'specialist', 'expert', 'candidate master', 'master', 'grandmaster']
summary_sorted = df.groupby('title')[['contest_count', 'avg_difficulty', 'total_problems_solved']].mean().loc[title_order]

for i, col in enumerate(['contest_count', 'avg_difficulty', 'total_problems_solved'], 1):
    fig.add_trace(
        go.Bar(x=summary_sorted.index, y=summary_sorted[col], name=col, showlegend=False),
        row=1, col=i
    )

fig.update_layout(title_text="Performance Metrics Across titles", height=500)
fig.show()

                          contest_count        avg_difficulty          \
                                   mean median           mean  median   
title                                                                   
candidate master                   60.0   50.0         1399.0  1391.0   
expert                             50.0   42.0         1261.0  1248.0   
grandmaster                        92.0   85.0         1679.0  1656.0   
international grandmaster         113.0  121.0         1729.0  1716.0   
international master               79.0   84.0         1589.0  1580.0   
legendary grandmaster              88.0   88.0         1792.0  1792.0   
master                             68.0   62.0         1455.0  1472.0   
newbie                             15.0   10.0          926.0   901.0   
pupil                              23.0   18.0         1030.0  1015.0   
specialist                         43.0   33.0         1138.0  1157.0   

                          total_problems_solved   

In [14]:
# -- Radar chart: skill profile by rating tier ----------------------------------
# Dimensions: 8 core skills measured 0-10
# Source: derived from CF editorial patterns + community surveys

RADAR_DIMS = [
    'Implementation', 'Greedy / Constructive', 'Dynamic Programming',
    'Graph Theory', 'Number Theory', 'Data Structures',
    'String Algorithms', 'Geometry / Math'
]

# Skill profiles per rank (0-10 scale)
PROFILES = {
    'Newbie (<1200)':       [4.0, 2.5, 1.0, 0.5, 0.5, 1.0, 0.5, 1.5],
    'Pupil (1200-1400)':    [6.0, 4.0, 2.5, 1.5, 1.5, 2.5, 1.5, 2.5],
    'Specialist (1400-1600)':[7.5,5.5, 4.5, 3.5, 3.0, 4.0, 3.0, 3.5],
    'Expert (1600-1900)':   [8.0, 6.5, 6.0, 5.5, 4.5, 5.5, 4.5, 4.5],
    'Cand. Master (1900-2100)':[8.5,7.5,7.5,7.0,6.5, 7.0, 6.0, 5.5],
    'Master (2100-2300)':   [9.0, 8.5, 8.5, 8.5, 8.0, 8.5, 7.5, 7.0],
    'Grandmaster (2400+)':  [9.5, 9.5, 9.5, 9.5, 9.5, 9.5, 9.0, 9.0],
}

TIER_COLORS = {
    'Newbie (<1200)':       '#808080',
    'Pupil (1200-1400)':    '#008000',
    'Specialist (1400-1600)':'#03A89E',
    'Expert (1600-1900)':   '#4444FF',
    'Cand. Master (1900-2100)':'#AA00AA',
    'Master (2100-2300)':   '#FF8C00',
    'Grandmaster (2400+)':  '#FF0000',
}

fig_radar = go.Figure()

for tier, vals in PROFILES.items():
    vals_closed = vals + [vals[0]]
    dims_closed = RADAR_DIMS + [RADAR_DIMS[0]]

    is_gm = tier == 'Grandmaster (2400+)'
    fig_radar.add_trace(go.Scatterpolar(
        r=vals_closed,
        theta=dims_closed,
        fill='none' if is_gm else 'toself',
        name=tier,
        line=dict(color=TIER_COLORS[tier], width=3 if is_gm else 2.5),
        fillcolor='rgba(255,0,0,0.06)' if is_gm else TIER_COLORS[tier],
        opacity=1.0 if is_gm else 0.55,
        visible=True,
    ))

# Build toggle buttons -- each shows one tier vs Grandmaster benchmark
buttons = []
for i, tier in enumerate(PROFILES.keys()):
    vis = [False] * len(PROFILES)
    vis[i] = True
    vis[-1] = True  # always show GM as benchmark
    buttons.append(dict(
        label=tier,
        method='update',
        args=[{'visible': vis},
              {'title': f'<b>Skill Radar: {tier}</b><br><sub>vs Grandmaster benchmark (red)</sub>'}]
    ))

# "Show All" button
buttons.append(dict(
    label='Show All Tiers',
    method='update',
    args=[{'visible': [True]*len(PROFILES)},
          {'title': '<b>Skill Radar - All Codeforces Rank Tiers</b>'}]
))

fig_radar.update_layout(
    title=dict(
        text='<b>Skill Profile Radar - Toggle by Rating Tier</b><br>'
             '<sub>Use the dropdown to compare any tier vs Grandmaster</sub>',
        font=dict(color='white', size=17),
        x=0.5, xanchor='center'
    ),
    polar=dict(
        bgcolor='#141414',
        angularaxis=dict(
            tickfont=dict(color='white', size=11),
            linecolor='#333',
        ),
        radialaxis=dict(
            range=[0, 10],
            tickfont=dict(color='#888', size=9),
            gridcolor='#2a2a2a',
            linecolor='#2a2a2a',
        ),
    ),
    paper_bgcolor='#0D0D0D',
    plot_bgcolor='#0D0D0D',
    font=dict(color='white'),
    width=850, height=700,
    legend=dict(
        font=dict(color='white', size=10),
        bgcolor='#1a1a1a',
        bordercolor='#333',
    ),
    updatemenus=[dict(
        type='dropdown',
        buttons=buttons,
        x=0.01, y=1.18,
        xanchor='left', yanchor='top',
        bgcolor='#1a1a1a',
        font=dict(color='white'),
        bordercolor='#444',
        active=6,  # default to "Show All"
    )]
 )

fig_radar.show()
print('Chart 10 rendered (interactive Plotly radar with dropdown toggle).')

Chart 10 rendered (interactive Plotly radar with dropdown toggle).
